# `transform.ipynb` — Funciones de limpieza y regex

Equivalente en notebook a **`transform.py`**. Concentra, en funciones
reutilizables (principio DRY), toda la lógica de limpieza y validación que
exige el documento de requerimientos funcionales:

| Función | Requisito | Qué hace |
|---|---|---|
| `parsear_importe` | RF-02 / RF-03 | Precios e importes sucios → `float` en EUR |
| `parsear_cliente` | RF-01 | `Cliente_Data` → nombre, apellidos, DNI validado |
| `parsear_fecha` | RF-03 | Fecha en 3 formatos distintos → ISO `YYYY-MM-DD` |
| `exportar_csv` | — | Volcado de respaldo de un DataFrame a `Resultados/` |
| `upsert_dataframe` | RF-06 | Carga idempotente en PostgreSQL (`ON CONFLICT DO UPDATE`) |
| `insertar_cuarentena` | RF-04 | Carga idempotente en `tb_errores_migracion` |

Cada función de validación sigue el mismo contrato: **nunca lanza una
excepción por un dato sucio**, siempre devuelve `(valor, motivo_error)` —
`motivo_error=None` si todo fue bien. Así, quien la llama decide qué hacer
(guardar el valor o mandarlo a cuarentena) sin try/except por cada fila.

Depende de `config.ipynb` (usa `TASA_USD_EUR`, `RE_DNI`, `DIR_RESULTADOS`),
así que lo primero que hace es cargarlo con `%run`.

## Cargar configuración y librerías

In [ ]:
%run config.ipynb


In [ ]:
import re
import logging
import pandas as pd
import numpy as np
from sqlalchemy import text

print('Librerías de transformación importadas correctamente')


## RF-02 / RF-03 — `parsear_importe`

Convierte un precio o importe sucio (`'220.00 USD'`, `'65.50 €'`, `'150,00'`,
`'-150.00 €'`, `'NULL'`...) a un `float` en EUR. Se reutiliza tanto para
`Precio_Compra`/`Precio_Venta` (RF-02) como para `MontoTotal_Raw` de ventas
(RF-03) — es la misma limpieza financiera en ambos casos.

Reglas: nulo/vacío/`'NULL'` → inválido · se limpian símbolos de divisa y se
normaliza la coma decimal a punto · no convertible a número → inválido ·
negativo → inválido · si venía en USD (`'USD'` o `'$'`), se convierte con
`TASA_USD_EUR` (1.15 USD = 1.00 EUR).

In [ ]:
def parsear_importe(valor, motivo_prefijo='Importe'):
    """RF-02 / RF-03 — Convierte un importe sucio ('220.00 USD', '65.50 €',
    '150,00', '-150.00 €', 'NULL'...) a un float en EUR.

    Reglas:
      - Nulo / vacío / literal 'NULL' -> inválido.
      - Quita símbolos de divisa y normaliza la coma decimal a punto.
      - Si no es convertible a número -> inválido.
      - Negativo -> inválido (RF-03).
      - Si venía en USD ('USD' o '$'), se convierte a EUR con TASA_USD_EUR.

    Devuelve (valor_eur: float|None, motivo_error: str|None).
    """
    if pd.isna(valor) or str(valor).strip() == '' or str(valor).strip().upper() == 'NULL':
        return None, f'{motivo_prefijo} nulo o vacío: {valor}'

    v = str(valor).strip().upper()
    es_usd = 'USD' in v or '$' in v
    numero_str = re.sub(r'[^0-9,.\-]', '', v).replace(',', '.')

    try:
        n = float(numero_str)
    except ValueError:
        return None, f'{motivo_prefijo} no numérico: {valor}'

    if n < 0:
        return None, f'{motivo_prefijo} negativo: {valor}'

    if es_usd:
        n = n / TASA_USD_EUR  # 1.15 USD = 1.00 EUR

    return round(n, 2), None

## RF-01 — `parsear_cliente`

Separa `Cliente_Data` (`"García Pérez, Juan - 12345678Z"`) en `nombre`,
`apellidos` y `dni` con una única expresión regular en la que el tramo
`- DNI` es **opcional**. Eso permite distinguir, con un motivo de rechazo
específico, tres casos distintos: formato totalmente irreconocible (p. ej.
`"UNKNOWN_USER"`), DNI ausente (`"Fernández, Ana"`) y DNI con formato
inválido (no son 8 dígitos + 1 letra).

In [ ]:
def parsear_cliente(cliente_data):
    """RF-01 — Separa 'Apellidos, Nombre - DNI' en sus tres componentes con
    regex y valida el DNI (8 dígitos + 1 letra). El tramo '- DNI' es opcional
    en la propia expresión regular para poder distinguir, con un motivo de
    rechazo específico, entre un cliente sin DNI y uno con formato irreconocible.

    Devuelve (nombre, apellidos, dni, motivo_error).
    """
    if pd.isna(cliente_data) or str(cliente_data).strip() == '':
        return None, None, None, f'Cliente_Data nulo o vacío: {cliente_data}'

    patron = re.compile(r'^\s*(?P<apellidos>[^,]+),\s*(?P<nombre>.+?)\s*(?:-\s*(?P<dni>\S+))?\s*$')
    m = patron.match(str(cliente_data))
    if not m:
        return None, None, None, f'Formato Cliente_Data no reconocido: {cliente_data}'

    nombre, apellidos, dni = m.group('nombre'), m.group('apellidos'), m.group('dni')
    if not dni:
        return nombre, apellidos, None, f'DNI ausente: {cliente_data}'
    if not RE_DNI.match(dni):
        return nombre, apellidos, dni, f'DNI con formato inválido: {dni}'

    return nombre, apellidos, dni.upper(), None


## RF-03 — `parsear_fecha`

Estandariza `Fecha` (que llega en `YYYY-MM-DD`, `DD/MM/YYYY`, `MM/DD/YYYY` o
el literal `INVALID_DATE`) al formato ISO. Los formatos con barras son
ambiguos cuando día y mes son ambos ≤ 12: se prioriza `DD/MM/YYYY`
(convención de empresa española, documentada también en la Memoria Técnica)
y, si esa interpretación no da una fecha válida, se reintenta como
`MM/DD/YYYY`.

In [ ]:
def parsear_fecha(valor):
    """RF-03 — Estandariza Fecha (YYYY-MM-DD, DD/MM/YYYY, MM/DD/YYYY o el
    literal 'INVALID_DATE') al formato ISO YYYY-MM-DD.

    Los formatos con barras son ambiguos cuando día y mes son ambos <= 12: se
    prioriza DD/MM/YYYY (convención de la empresa, española) y, si esa
    interpretación no es una fecha válida, se prueba MM/DD/YYYY como fallback.
    Esta prioridad es una decisión de negocio documentada en la Memoria Técnica.

    Devuelve (fecha: date|None, motivo_error: str|None).
    """
    if pd.isna(valor) or str(valor).strip() == '' or str(valor).strip().upper() == 'INVALID_DATE':
        return None, f'Fecha inválida: {valor}'

    v = str(valor).strip()
    try:
        if re.match(r'^\d{4}-\d{2}-\d{2}$', v):
            return pd.to_datetime(v, format='%Y-%m-%d').date(), None
        if re.match(r'^\d{1,2}/\d{1,2}/\d{4}$', v):
            try:
                return pd.to_datetime(v, format='%d/%m/%Y').date(), None
            except ValueError:
                return pd.to_datetime(v, format='%m/%d/%Y').date(), None
    except ValueError:
        pass
    return None, f'Formato de fecha no reconocido: {v}'


## Funciones de carga (RF-04 / RF-06)

Tres funciones más, ya orientadas a la fase de **Load** pero que viven aquí
por el mismo principio DRY: se reutilizan igual para las 6 tablas de
dimensión/hechos (`upsert_dataframe`) y para la cuarentena
(`insertar_cuarentena`), así que tiene sentido definirlas una única vez junto
al resto de funciones de limpieza.

- **`exportar_csv`** — respaldo de cualquier DataFrame en `Resultados/*.csv`.
- **`upsert_dataframe`** — `INSERT ... ON CONFLICT (clave_origen) DO UPDATE`:
  es lo que hace idempotente la carga (RF-06), reejecutar el ETL actualiza en
  lugar de duplicar.
- **`insertar_cuarentena`** — misma idea aplicada a `tb_errores_migracion`,
  con `DO NOTHING` sobre `(tabla_origen, datos_raw, motivo_rechazo)` para no
  duplicar el mismo error en cada re-ejecución (RF-04 + RF-06).

In [ ]:
def exportar_csv(df, nombre, carpeta=DIR_RESULTADOS):
    """Guarda un DataFrame como CSV en Resultados/, con manejo de errores."""
    ruta = os.path.join(carpeta, nombre)
    try:
        df.to_csv(ruta, index=False, encoding='utf-8')
        logging.info(f'CSV exportado: {nombre} ({len(df)} filas)')
        print(f'   Guardado: {nombre} ({len(df)} filas)')
    except Exception as e:
        logging.error(f'Error exportando {nombre}: {e}')
        print(f'   Error exportando {nombre}: {e}')


def upsert_dataframe(df, tabla, columnas_conflicto, engine, chunksize=500):
    """Carga un DataFrame en PostgreSQL de forma IDEMPOTENTE (RF-06):
    sube el DataFrame a una tabla temporal y hace
    INSERT ... ON CONFLICT (columnas_conflicto) DO UPDATE, de modo que
    reejecutar el ETL actualiza las filas existentes en lugar de duplicarlas.
    Devuelve el nº de filas procesadas (0 si hay error o el DataFrame está vacío).
    """
    if df is None or df.empty:
        logging.warning(f'{tabla}: DataFrame vacío, no se carga nada')
        return 0
    try:
        with engine.begin() as conn:
            df.to_sql('tmp_stage_' + tabla, conn, if_exists='replace', index=False, chunksize=chunksize)
            columnas = list(df.columns)
            cols_str = ', '.join(columnas)
            update_str = ', '.join(f'{c}=EXCLUDED.{c}' for c in columnas if c not in columnas_conflicto)
            conflict_str = ', '.join(columnas_conflicto)
            conn.execute(text(f"""
                INSERT INTO {tabla} ({cols_str})
                SELECT {cols_str} FROM tmp_stage_{tabla}
                ON CONFLICT ({conflict_str}) DO UPDATE SET {update_str};
            """))
            conn.execute(text(f'DROP TABLE tmp_stage_{tabla}'))
        logging.info(f'Cargada {tabla}: {len(df)} registros (upsert)')
        print(f'   PostgreSQL: {tabla} -> {len(df)} registros (upsert)')
        return len(df)
    except Exception as e:
        logging.error(f'Error cargando {tabla}: {e}')
        print(f'   Error cargando {tabla}: {e}')
        return 0


def insertar_cuarentena(df_errores, engine):
    """RF-04 — Inserta los errores de validación en tb_errores_migracion.
    La restricción UNIQUE (tabla_origen, datos_raw, motivo_rechazo) definida en
    el DDL, junto con ON CONFLICT DO NOTHING, evita duplicar el mismo error en
    cada re-ejecución (RF-06).
    """
    if df_errores.empty:
        return 0
    try:
        with engine.begin() as conn:
            df_errores.to_sql('tmp_stage_errores', conn, if_exists='replace', index=False)
            conn.execute(text("""
                INSERT INTO tb_errores_migracion (tabla_origen, datos_raw, motivo_rechazo, fecha_deteccion)
                SELECT tabla_origen, datos_raw, motivo_rechazo, fecha_deteccion FROM tmp_stage_errores
                ON CONFLICT (tabla_origen, datos_raw, motivo_rechazo) DO NOTHING;
            """))
            conn.execute(text('DROP TABLE tmp_stage_errores'))
        logging.info(f'Cuarentena: {len(df_errores)} filas evaluadas para tb_errores_migracion')
        print(f'   Cuarentena: {len(df_errores)} filas evaluadas (duplicados ya existentes se ignoran)')
        return len(df_errores)
    except Exception as e:
        logging.error(f'Error insertando en cuarentena: {e}')
        print(f'   Error insertando en cuarentena: {e}')
        return 0


## Confirmación

In [ ]:
print('--- transform.ipynb cargado correctamente ---')
print('Funciones disponibles: parsear_importe, parsear_cliente, parsear_fecha,')
print('exportar_csv, upsert_dataframe, insertar_cuarentena')
